# LAB05 - Mineral Prospectivity Mapping

**Public portfolio notebook**

This notebook is a reproducible scientific walkthrough of the validated LAB05
workflow. It does not reimplement or retrain the production experiment.

Instead, it reads the persisted artifacts produced by Gates 0-6, checks the
frozen headline metrics, and presents the evidence used to support the public
portfolio conclusions.

All mineral-system, geochemical, occurrence, and prospectivity data used here
are synthetic.


## Scientific question

Can spatially validated machine learning integrate geological, structural,
geophysical, and compositional geochemical evidence to rank mineral
prospectivity while minimizing spatial leakage and exploration area?

The validated experiment uses:

- EPSG:32719;
- an 80 km x 80 km synthetic study area;
- a 1 km grid with 6,400 cells;
- 100 known synthetic occurrences;
- 115 sealed hidden deposits;
- spatially grouped cross-validation;
- positive-unlabeled background sampling;
- Average Precision as the primary model-selection metric;
- a sealed hidden-oracle evaluation only after model freeze.


In [1]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lab05_mineral_prospectivity":
    candidate = Path("labs/lab05_mineral_prospectivity")
    if candidate.exists():
        LAB_DIR = candidate.resolve()

RESULTS_DIR = LAB_DIR / "results"
README_PATH = LAB_DIR / "README.md"

required = [
    RESULTS_DIR / "benchmark_oof_aggregate_metrics.csv",
    RESULTS_DIR / "benchmark_oof_metrics.csv",
    RESULTS_DIR / "success_rate_curve_oof.csv",
    RESULTS_DIR / "prospectivity_ranking_summary_oos.csv",
    RESULTS_DIR / "prospectivity_scores_oos.csv",
    README_PATH,
]

missing = [str(path) for path in required if not path.exists()]
assert not missing, f"Missing validated artifacts: {missing}"

print(f"LAB_DIR: {LAB_DIR}")
print("Validated artifact inventory: PASS")


LAB_DIR: /mnt/c/GeoAI/geoai-mineral-labs/labs/lab05_mineral_prospectivity
Validated artifact inventory: PASS


## Frozen experimental design

Feature families were frozen before hidden-oracle evaluation.

The selected model was determined only from the predeclared primary criterion:
mean pooled spatial out-of-fold Average Precision.

Random stratified cross-validation is retained only as a diagnostic comparison;
it is not used to select the final model.


In [2]:
benchmark = pd.read_csv(
    RESULTS_DIR / "benchmark_oof_aggregate_metrics.csv"
)

print("Aggregate benchmark shape:", benchmark.shape)
display(benchmark)


Aggregate benchmark shape: (18, 10)


,validation_strategy,feature_set,model,n_replicates,average_precision_mean,average_precision_std,average_precision_median,roc_auc_mean,roc_auc_std,roc_auc_median
0,random,A,dummy,10,0.166667,2.925695e-17,0.166667,0.500000,0.000000,0.50000
1,random,A,logistic_regression,10,0.372654,3.892523e-02,0.368160,0.727044,0.015237,0.72473
2,random,A,random_forest,10,0.377841,5.133747e-02,0.381560,0.725836,0.021052,0.73475
3,random,B,dummy,10,0.166667,2.925695e-17,0.166667,0.500000,0.000000,0.50000
4,random,B,logistic_regression,10,0.347355,4.012330e-02,0.338000,0.707964,0.020448,0.70647
5,random,B,random_forest,10,0.380612,2.868427e-02,0.382045,0.741667,0.013304,0.74270
6,random,C,dummy,10,0.166667,2.925695e-17,0.166667,0.500000,0.000000,0.50000
7,random,C,logistic_regression,10,0.366768,4.143158e-02,0.355556,0.733184,0.016036,0.73346
8,random,C,random_forest,10,0.366755,3.381704e-02,0.355628,0.719808,0.016529,0.71835
9,spatial,A,dummy,10,0.158175,2.786597e-03,0.157384,0.473204,0.008933,0.47103


## Model-selection result

The frozen selection is **Feature A + Logistic Regression**.

Validated spatial pooled out-of-fold metrics:

- mean Average Precision: 0.393412;
- median Average Precision: 0.380985;
- mean ROC-AUC: 0.734202.

Competing configurations were close. Secondary metrics did not replace the
predeclared primary selection rule.


In [3]:
readme = README_PATH.read_text(encoding="utf-8")

headline_metrics = {
    "mean_spatial_oof_ap": "0.393412",
    "median_spatial_oof_ap": "0.380985",
    "mean_spatial_oof_roc_auc": "0.734202",
}

for name, value in headline_metrics.items():
    assert value in readme, f"{name} is inconsistent with the validated README"

print("Frozen model-selection metrics: PASS")


Frozen model-selection metrics: PASS


## Out-of-sample prospectivity ranking

After model selection, LAB05 generates ranking scores for the 6,300 unlabeled
candidate cells.

A candidate cell is scored only by replicate models for which that cell was not
sampled as training background. The resulting `prospectivity_score` is an
ensemble ranking score, **not a calibrated deposit probability**.


In [4]:
ranking_summary = pd.read_csv(
    RESULTS_DIR / "prospectivity_ranking_summary_oos.csv"
)
scores = pd.read_csv(
    RESULTS_DIR / "prospectivity_scores_oos.csv"
)

print("Ranking summary shape:", ranking_summary.shape)
print("Candidate score table shape:", scores.shape)
display(ranking_summary.head(20))


Ranking summary shape: (3, 6)
Candidate score table shape: (6400, 21)


,tier,candidate_count,expected_count,mean_oos_score,median_oos_score,mean_oos_support
0,top_1pct,63,63,0.600631,0.601388,9.238095
1,top_5pct,315,315,0.539161,0.534976,9.123810
2,top_10pct,630,630,0.468628,0.461688,9.161905


In [5]:
def find_column(frame, exact=(), contains=()):
    lookup = {column.lower(): column for column in frame.columns}

    for candidate in exact:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    for column in frame.columns:
        low = column.lower()
        if all(token.lower() in low for token in contains):
            return column

    return None


x_col = find_column(scores, exact=("Easting", "easting", "x"))
y_col = find_column(scores, exact=("Northing", "northing", "y"))

score_col = find_column(
    scores,
    exact=(
        "prospectivity_score",
        "prospectivity_score_mean",
        "mean_score",
        "score_mean",
    ),
)

if score_col is None:
    score_col = find_column(scores, contains=("score", "mean"))

if x_col and y_col and score_col:
    plot_data = scores[[x_col, y_col, score_col]].dropna()

    fig, ax = plt.subplots(figsize=(8, 7))
    scatter = ax.scatter(
        plot_data[x_col],
        plot_data[y_col],
        c=plot_data[score_col],
        s=12,
    )
    ax.set_title("LAB05 out-of-sample prospectivity ranking")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    fig.colorbar(scatter, ax=ax, label=score_col)
    plt.show()
else:
    print(
        "Map preview skipped because coordinate/score column names "
        "could not be resolved safely."
    )


/tmp/ipykernel_384/3914604107.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sealed hidden-oracle evaluation

The hidden oracle remained sealed until after model selection and ranking logic
were frozen.

Validated hidden-oracle metrics:

| Metric | Result |
| --- | ---: |
| Hidden deposits | 115 |
| Hidden Average Precision | 0.056650 |
| Hidden ROC-AUC | 0.709070 |
| Hidden captured in top 1% | 6 / 115 |
| Lift in top 1% | 5.22x |
| Hidden captured in top 5% | 25 / 115 |
| Lift in top 5% | 4.35x |
| Hidden captured in top 10% | 44 / 115 |
| Lift in top 10% | 3.83x |
| Median hidden rank | 1117 / 6300 |

The hidden Average Precision must be interpreted relative to the very low
hidden-deposit prevalence. The useful result is enrichment and search-space
reduction, not a claim of calibrated deposit detection.


In [6]:
hidden_metrics = [
    "0.056650",
    "0.709070",
    "5.22x",
    "4.35x",
    "3.83x",
]

for value in hidden_metrics:
    assert value in readme, f"Hidden-oracle metric missing from validated README: {value}"

print("Frozen hidden-oracle metrics: PASS")


Frozen hidden-oracle metrics: PASS


## Validation and reproducibility

The public LAB05 milestone was validated with:

- 163 passing project tests at the v0.2.0 checkpoint;
- pre-commit quality controls;
- Gitleaks secret scanning;
- signed milestone commits;
- a signed v0.2.0 release tag;
- public synthetic data only.

This notebook is intentionally downstream of the validated workflow. Its role
is communication, inspection, and reproducible review of persisted evidence.


In [7]:
assert "163 passed" in readme
assert "not a" in readme.lower() and "calibrated" in readme.lower()

print("Public documentation consistency: PASS")


Public documentation consistency: PASS


## Limitations

1. The entire mineral system is synthetic.
2. Background cells are unlabeled, not confirmed negatives.
3. Scores rank prospectivity; they are not calibrated occurrence probabilities.
4. Spatial validation is methodologically required even though random-CV
   inflation was not empirically demonstrated in every LAB05 comparison.
5. The workflow does not replace field mapping, drilling, geophysical
   acquisition, laboratory assays, or competent geological interpretation.
6. Hidden-oracle performance is evidence for this frozen synthetic experiment,
   not a guarantee of real-world exploration performance.


## Conclusion

LAB05 demonstrates an end-to-end mineral prospectivity workflow in which
spatial validation, leakage control, compositional evidence, out-of-fold
evaluation, out-of-sample ranking, model freeze, and hidden-oracle evaluation
are treated as part of the scientific design.

The notebook presents the validated evidence without introducing a second
implementation of the modelling pipeline.
